# 1- Initialize Spark Session

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Streaming_flights_lab5") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

spark

# 2- Define schema for the CSV files


In [6]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("DEST_COUNTRY_NAME", StringType(), True),
    StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
    StructField("count", IntegerType(), True)
])
print(schema)

StructType(List(StructField(DEST_COUNTRY_NAME,StringType,true),StructField(ORIGIN_COUNTRY_NAME,StringType,true),StructField(count,IntegerType,true)))


# 3-Read stream from the CSV directory inside the container


In [7]:
# We use the file:// scheme to specify the local path inside the container
input_path = "file:///data/Lab-5/csv"

df = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load(input_path)

# 4- Aggregate the data

In [8]:
# Aggregate the data on DEST_COUNTRY_NAME and ORIGIN_COUNTRY_NAME
from pyspark.sql.functions import sum

df_aggregated = df.groupBy("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME") \
    .agg(sum("count").alias("total_count"))

# 5- Writing Output to the Console

In [9]:
# Start the streaming query writing output to the console
# We use Update output mode to show only records changed in each micro-batch
query = df_aggregated.writeStream \
    .format("console") \
    .outputMode("update") \
    .start()

In [10]:
# Run the query for some time to let Spark process the existing files, then stop
import time
time.sleep(20)

query.stop()
query.awaitTermination()
print("Streaming query stopped successfully")

Streaming query stopped successfully
